# Processing datasets for FHIR retrieval benchmark

Install the package from the repository root before running this notebook:

```bash
pip install -e .
```

This notebook transforms the raw upstream datasets into the bench's parquet format. It requires the raw source files for each dataset (see `docs/docs/datasets.md` for where to obtain them); it cannot be run from the packaged repository alone.

In [ ]:
import os
import json
import re
import datetime
import time
import pickle
import uuid
from typing import Any, List, Optional, Tuple

import pandas as pd
import numpy as np
import requests
import tqdm
import yaml
from IPython.display import display, HTML

from fhir_retrieval_bench.data import fhir_utils

In [ ]:
OUTPUT_DIR = "data/" # @param {type: "string"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# FHIRAgentBench

In [ ]:
# @title Load QA pairs
start = time.time()
qa_path = "data/fhiragentbench/questions_answers_sql_fhir.csv"
if not os.path.exists(qa_path):
  print(f"WARNING: {qa_path} not found. Please place your raw dataset file there.")
else:
  with open(qa_path, "r") as f:
    fhiragentbench_qa = pd.read_csv(f)
  print(f"\n✅ Done. Reading the file took {time.time() - start:.4f} seconds.")
  print("qa count: ", fhiragentbench_qa.shape)

In [ ]:
fhiragentbench_qa.head()

In [ ]:
# @title Load FHIR bundle
# If you are curious about how this file was created, refer to:
# notebooks/convert_gcp_fhir_store_to_json.ipynb (reference)
start = time.time()
fhir_path = "data/fhiragentbench/mimic_consumer_rewritten_queries_all_patients_with_fhir.pq"
if not os.path.exists(fhir_path):
  print(f"WARNING: {fhir_path} not found. Please place your raw dataset file there.")
else:
  with open(fhir_path, "rb") as f:
    fhiragentbench_fhir = pd.read_parquet(f)
  print(f"\n✅ Done. Reading the file took {time.time() - start:.4f} seconds.")

In [ ]:
# @title Extract unique FHIR bundle

# get # of unique FHIR bundles for each patient id
fhiragentbench_fhir_unique_count = fhiragentbench_fhir.groupby("Patient ID")[
    "FHIR Bundle"
].nunique()

if not (fhiragentbench_fhir_unique_count == 1).all():
  raise RuntimeError("FHIR bundles from the patients are not identical!!")

fhiragentbench_unique_fhir = (
    fhiragentbench_fhir.groupby("Patient ID")
    .first()
    .reset_index()[["Patient ID", "FHIR Bundle"]]
)
fhiragentbench_unique_fhir["FHIR Bundle"] = fhiragentbench_unique_fhir["FHIR Bundle"].map(json.loads)

print("patient count: ", fhiragentbench_unique_fhir.shape)

In [ ]:
# @title Transform bundle to standard format

def transform_fhiragentgench_bundle(raw_data: dict[str, Any]):
  bundle: dict[str, Any] = {
      "resourceType": "Bundle",
      "type": "collection",
      "entry": [],
  }
  for key, resources in raw_data.items():
    if isinstance(resources, list):
      for resource in resources:
        if isinstance(resource, dict):
          if "resourceType" not in resource:
            resource["resourceType"] = key
          bundle["entry"].append({"resource": resource})
  if fhir_utils.verify_with_pydantic(bundle):
    return bundle
  else:
    raise RuntimeError("Transform failure")

fhiragentbench_unique_fhir["FHIR Bundle"] = fhiragentbench_unique_fhir["FHIR Bundle"].map(transform_fhiragentgench_bundle).map(json.dumps)

In [ ]:
# @title Verify the overlap between QA and FHIR
if not set(fhiragentbench_qa["patient_fhir_id"]) == set(fhiragentbench_unique_fhir["Patient ID"]):
  raise RuntimeError("QA pairs and FHIR bundles do not match")

In [ ]:
# @title Save data
os.makedirs(os.path.join(OUTPUT_DIR, "fhiragentbench"), exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "fhiragentbench/qa.csv"), "w") as f:
  fhiragentbench_qa.to_csv(f)
with open(os.path.join(OUTPUT_DIR, "fhiragentbench/fhir_bundles.pq"), "wb") as f:
  fhiragentbench_unique_fhir.to_parquet(f)

# EHRQA

In [ ]:
# @title Load QA pairs
start = time.time()
qa_path = "data/ehrqa/validation.json"
if not os.path.exists(qa_path):
  print(f"WARNING: {qa_path} not found. Please place your raw dataset file there.")
else:
  with open(qa_path, "r") as f:
    ehrqa_qa = json.load(f)
  print(f"\n✅ Done. Reading the file took {time.time() - start:.4f} seconds.")
  print("size of qa: ", len(ehrqa_qa))

In [ ]:
# @title Load FHIR bundle
start = time.time()
fhir_path = "data/ehrqa/dictionary_of_synthea_patients_for_ehrqa_validation.pkl"
if not os.path.exists(fhir_path):
  print(f"WARNING: {fhir_path} not found. Please place your raw dataset file there.")
else:
  with open(fhir_path, "rb") as f:
    ehrqa_fhir = pickle.load(f)
  print(f"\n✅ Done. Reading the file took {time.time() - start:.4f} seconds.")

In [ ]:
# @title Verify json integrity and convert to parquet)
# checking json synthax. should run without issue.
for patient_id in ehrqa_fhir.keys():
  json.loads(ehrqa_fhir[patient_id])

ehrqa_fhir_pq = pd.Series(ehrqa_fhir).to_frame(name="FHIR Bundle").rename_axis("Patient ID").reset_index()
ehrqa_fhir_pq["FHIR Bundle"] = ehrqa_fhir_pq["FHIR Bundle"].map(json.loads)

print("patient count: ", ehrqa_fhir_pq.shape)

In [ ]:
# @title Transform bundle to standard format

def transform_ehrqa_bundle(raw_data: dict[str, Any]):
  bundle: dict[str, Any] = {
      "resourceType": "Bundle",
      "type": "collection",
      "entry": [],
  }
  for key, resources in raw_data.items():
    if isinstance(resources, list):
      for resource in resources:
        if isinstance(resource, dict):
          if "resourceType" not in resource:
            resource["resourceType"] = key
          bundle["entry"].append({"resource": resource})
  if fhir_utils.verify_with_pydantic(bundle):
    return bundle
  else:
    raise RuntimeError("Transform failure")

# fhiragentbench_unique_fhir["FHIR Bundle"] =
ehrqa_fhir_pq["FHIR Bundle"] = ehrqa_fhir_pq["FHIR Bundle"].map(transform_ehrqa_bundle).map(json.dumps)

In [ ]:
# @title Verify the overlap between QA and FHIR
if not set([qa["patient_id"] for qa in ehrqa_qa]) == set(ehrqa_fhir_pq["Patient ID"]):
  raise RuntimeError("QA pairs and FHIR bundles do not match")

In [ ]:
# @title Save data
os.makedirs(os.path.join(OUTPUT_DIR, "ehrqa"), exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "ehrqa/qa.pickle"), "wb") as f:
  pickle.dump(ehrqa_qa, f)
with open(os.path.join(OUTPUT_DIR, "ehrqa/fhir_bundles.pq"), "wb") as f:
  ehrqa_fhir_pq.to_parquet(f)

# MedAgentBench

In [ ]:
# @title Load QA pairs

# Send a GET request to the URL
response = requests.get("https://raw.githubusercontent.com/stanfordmlgroup/MedAgentBench/refs/heads/main/data/medagentbench/test_data_v2.json")

if response.status_code == 200:
    data = response.json()
    print(f"Loaded {len(data)} items.")
    print(data[0]) # Print the first element to verify
else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")

medagentbench_qa_df = pd.DataFrame(data)
medagentbench_qa_df

print("original size of qa: ", len(medagentbench_qa_df))

In [ ]:
# @title Load FHIR bundle
import glob

# If you are curious about how this file was created, refer to:
# scripts/medagent_scape.py (reference)

fhir_json_files = glob.glob("data/medagentbench/patients/*.json")

def load_medagentbench_json_data(json_path):
  """Loads a single JSON file"""
  with open(json_path, "r") as f:
    data = json.load(f)
    return {
            "Patient ID": data["Patient"]["id"],
            "FHIR Bundle": data,
        }

print(f"Loading files...")
medagentbench_fhir = []
if not fhir_json_files:
  print("WARNING: No patient JSON files found under data/medagentbench/patients/")
else:
  for path in tqdm.tqdm(fhir_json_files):
    medagentbench_fhir.append(load_medagentbench_json_data(path))

medagentbench_fhir_df = pd.DataFrame(medagentbench_fhir)

In [ ]:
[len(str(i)) for i in medagentbench_fhir]

In [ ]:
medagentbench_fhir_df

In [ ]:
# @title Compute ground truth

def _parse_medagent_timestamp(context: str) -> Optional[datetime.datetime]:
  """Extracts the reference timestamp from MedAgentBench context string."""
  match = re.search(r"It's\s+(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\+\d{2}:\d{2})", context)
  if match:
    return datetime.datetime.fromisoformat(match.group(1))
  return None


def _get_observations_by_code(
    raw_data: dict[str, Any], code: str,
) -> List[Tuple[datetime.datetime, float]]:
  """Returns (datetime, value) pairs for observations matching the given code."""
  results = []
  for obs in raw_data.get("Records", {}).get("Observation", []):
    codings = obs.get("code", {}).get("coding", [])
    if not any(c.get("code") == code for c in codings):
      continue
    dt_str = obs.get("effectiveDateTime")
    val_q = obs.get("valueQuantity", {})
    value = val_q.get("value")
    if dt_str is None or value is None:
      continue
    try:
      dt = datetime.datetime.fromisoformat(dt_str)
      results.append((dt, float(value)))
    except (ValueError, TypeError):
      continue
  return results


def _compute_medagent_ground_truth(
    task: dict[str, Any], raw_data: dict[str, Any] | None,
) -> str:
  """Computes ground truth for MedAgentBench QA tasks from patient data.

  Handles:
    task1: MRN lookup (already has sol in the data).
    task2: Patient age from birthDate.
    task4: Most recent magnesium level within 24h.
    task6: Average CBG over last 24h.
    task7: Most recent CBG (no time constraint).
  """
  task_id = task["id"]
  task_type = task_id.split("_")[0]
  context = task["context"]

  # task1 already has sol.
  if task_type == "task1":
    if len(task["sol"])==1:
      return task["sol"][0]
    else:
      print(f"{task_type} sol error", task)
      return None

  # task2: age = floor(ref_timestamp - birthDate)
  elif task_type == "task2":
    ref_time = _parse_medagent_timestamp(context)
    birth_str = raw_data.get("Patient", {}).get("birthDate")
    if ref_time and birth_str:
      birth_date = datetime.date.fromisoformat(birth_str)
      ref_date = ref_time.date()
      age = ref_date.year - birth_date.year
      if (ref_date.month, ref_date.day) < (birth_date.month, birth_date.day):
        age -= 1
      return str(age)
    else:
      print(f"{task_type} skip", ref_time, birth_str, task)
      return None

  # task4: most recent magnesium within 24h
  elif task_type == "task4":
    ref_time = _parse_medagent_timestamp(context)
    if not ref_time:
      print(f"{task_type} skipped; no ref time", task)
      return None
    obs = _get_observations_by_code(raw_data, "MG")
    cutoff = ref_time - datetime.timedelta(hours=24)
    recent = [(dt, v) for dt, v in obs if cutoff <= dt <= ref_time]
    if not recent:
      return "-1"
    recent.sort(key=lambda x: x[0], reverse=True)
    return str(recent[0][1])

  # task6: average CBG over last 24h
  elif task_type == "task6":
    ref_time = _parse_medagent_timestamp(context)
    if not ref_time:
      print(f"{task_type} skip", task)
      return None
    obs = _get_observations_by_code(raw_data, "GLU")
    cutoff = ref_time - datetime.timedelta(hours=24)
    recent = [v for dt, v in obs if cutoff <= dt <= ref_time]
    if not recent:
      return "-1"
    avg = sum(recent) / len(recent)
    # Round to 1 decimal place to avoid floating point noise.
    return str(round(avg, 1))

  # task7: most recent CBG (no time constraint)
  elif task_type == "task7":
    obs = _get_observations_by_code(raw_data, "GLU")
    if not obs:
      return "No record found"
    obs.sort(key=lambda x: x[0], reverse=True)
    return str(obs[0][1])

  print(f"{task_type} is not supported", task)
  return None

def qa_row_mapper(row):

  fhir_bundle_mapping = medagentbench_fhir_df.set_index("Patient ID")['FHIR Bundle'].to_dict()
  fhir_bundle = fhir_bundle_mapping.get(row["eval_MRN"])

  return _compute_medagent_ground_truth(row.to_dict(), fhir_bundle)


medagentbench_qa_df["ground_truth"] = (medagentbench_qa_df.apply(qa_row_mapper, axis=1))
print("Original shape", medagentbench_qa_df.shape)

medagentbench_qa_df = medagentbench_qa_df[~medagentbench_qa_df["ground_truth"].isnull()]
print("After filtering null GT", medagentbench_qa_df.shape)

medagentbench_qa_df = medagentbench_qa_df[~medagentbench_qa_df["eval_MRN"].isnull()]
print("After filtering null patient id", medagentbench_qa_df.shape)

print("# records with ground truth", medagentbench_qa_df.shape)

In [ ]:
# @title Transform bundle to standard format

def transform_medagentbench_bundle(raw_data: dict[str, Any]):
  bundle: dict[str, Any] = {
      "resourceType": "Bundle",
      "type": "collection",
      "entry": [],
  }

  assert set(raw_data.keys()) == set(['Patient', 'Records']), str(raw_data)

  bundle["entry"].append(
    {
        "resource": raw_data["Patient"]
    }
  )
  for resource_type, resources in raw_data["Records"].items():
    for resource in resources:
      bundle["entry"].append(
        {
            "resource": resource
        }
      )

  if fhir_utils.verify_with_pydantic(bundle):
    return bundle
  else:
    raise RuntimeError("Transform failure")

# fhiragentbench_unique_fhir["FHIR Bundle"] =
medagentbench_fhir_df["FHIR Bundle"] = medagentbench_fhir_df["FHIR Bundle"].map(transform_medagentbench_bundle).map(json.dumps)

In [ ]:
# @title Verify the overlap between QA and FHIR
if not set(medagentbench_qa_df["eval_MRN"].fillna(-9)).issubset(set(medagentbench_fhir_df["Patient ID"])):
  raise RuntimeError("QA pairs have missing FHIR bundles")

In [ ]:
medagentbench_fhir_df["Patient ID"]

In [ ]:
medagentbench_qa_df[medagentbench_qa_df["eval_MRN"].fillna("-9")=="-9"]

In [ ]:
# @title Save data
os.makedirs(os.path.join(OUTPUT_DIR, "medagentbench"), exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "medagentbench/qa.pickle"), "wb") as f:
  medagentbench_qa_df.to_pickle(f)
with open(os.path.join(OUTPUT_DIR, "medagentbench/fhir_bundles.pq"), "wb") as f:
  medagentbench_fhir_df.to_parquet(f)

# FHIRPath-QA

In [ ]:
data/fhirpathqa/qa.pickle
data/fhirpathqa/fhir_bundles.pq

In [ ]:
# @title Load QA pairs

qa_path = "data/fhirpathqa/qa.pickle"
if not os.path.exists(qa_path):
  print(f"WARNING: {qa_path} not found. Please place your raw dataset file there.")
else:
  with open(qa_path, "rb") as f:
    fhirpathqa_qa = pickle.load(f)
  fhirpathqa_qa_df = pd.DataFrame(fhirpathqa_qa)
  print("original size of qa: ", len(fhirpathqa_qa_df))

In [ ]:
fhirpathqa_qa[0]

In [ ]:
# @title Load FHIR bundle

fhir_path = "data/fhirpathqa/fhir_bundles.pq"
if not os.path.exists(fhir_path):
  print(f"WARNING: {fhir_path} not found. Please place your raw dataset file there.")
else:
  with open(fhir_path, "rb") as f:
    fhirpathqa_fhir = pd.read_parquet(f)
  print(fhirpathqa_fhir.shape)

In [ ]:
# @title Verify json integrity
assert fhirpathqa_fhir["FHIR Bundle"].map(json.loads).map(fhir_utils.verify_with_pydantic).all()

In [ ]:
# @title Verify the overlap between QA and FHIR
if not set(fhirpathqa_qa_df["patient_id"]) == set(fhirpathqa_fhir["Patient ID"]):
  raise RuntimeError("QA pairs and FHIR bundles do not match")

In [ ]:
# @title Save data
os.makedirs(os.path.join(OUTPUT_DIR, "fhirpathqa"), exist_ok=True)

with open(os.path.join(OUTPUT_DIR, "fhirpathqa/qa.pickle"), "wb") as f:
  pickle.dump(fhirpathqa_qa, f)

with open(os.path.join(OUTPUT_DIR, "fhirpathqa/fhir_bundles.pq"), "wb") as f:
  fhirpathqa_fhir.to_parquet(f)

# Change permission of the data folder